# Tratamento e Normalização dos dados

# 1. Carregamento e visão geral dos dados
## 1.1 Carregamento da base

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/bruto/balneabilidade_bruto.csv")

print(f"Dimensões da base: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

display(df.head())

## 1.2 Estrutura e tipos das colunas

In [ ]:
tipos = pd.DataFrame({
    "coluna": df.columns,
    "tipo_atual": df.dtypes.astype(str),
    "qtd_valores": df.notna().sum().values,
    "qtd_nulos": df.isna().sum().values
})

display(tipos)

,coluna,tipo_atual,qtd_valores,qtd_nulos
analise_id,analise_id,int64,1694,0
quantitativo,quantitativo,int64,1694,0
analise_data,analise_data,object,1694,0
trecho_id,trecho_id,int64,1694,0
trecho_nome,trecho_nome,object,1694,0
trecho_descricao,trecho_descricao,object,1694,0
estacao,estacao,object,1694,0
latitude,latitude,float64,1694,0
longitude,longitude,float64,1694,0
periodicidade,periodicidade,int64,1694,0


In [ ]:
# converter a coluna analise_data para datetime
df["analise_data"] = pd.to_datetime(
    df["analise_data"],
    errors="coerce"
)

## 1.3 Valores ausentes

In [ ]:
print(df["analise_data"].dtype)
print("Datas inválidas:", df["analise_data"].isna().sum())

datetime64[ns]
Datas inválidas: 0


## 1.4 Cardinalidade

In [ ]:
cardinalidade = pd.DataFrame({
    "coluna": df.columns,
    "qtd_unicos": [
        df[col].nunique(dropna=False)
        for col in df.columns
    ],
    "qtd_unicos_sem_nulo": [
        df[col].nunique()
        for col in df.columns
    ]
})

display(cardinalidade)

,coluna,qtd_unicos,qtd_unicos_sem_nulo
0,analise_id,1694,1694
1,quantitativo,317,317
2,analise_data,1526,1526
3,trecho_id,74,74
4,trecho_nome,74,74
5,trecho_descricao,71,71
6,estacao,74,74
7,latitude,74,74
8,longitude,74,74
9,periodicidade,2,2


A coluna `analise_id` apresenta 1.694 valores únicos para 1.694 registros, indicando que cada linha possui um identificador distinto. Esse comportamento é consistente com a utilização da coluna como identificador das análises. A coluna `quantitativo` apresenta 317 valores distintos, o que é esperado, uma vez que diferentes análises podem apresentar o mesmo resultado.

A coluna `analise_data` possui 1.526 valores distintos, indicando a ocorrência de datas repetidas. Esse comportamento é possível, pois diferentes análises podem ter sido realizadas no mesmo instante ou em momentos muito próximos.

A coluna `trecho_id` apresenta 74 valores únicos, correspondentes aos 74 trechos identificados na base. A coluna `trecho_nome` também possui 74 valores distintos, mantendo correspondência com os identificadores de trecho. Já `trecho_descricao` apresenta 71 valores únicos para os 74 trechos, indicando que existem descrições compartilhadas ou repetidas entre alguns trechos.

As colunas `estacao`, `latitude` e `longitude` apresentam 74 valores distintos, o que indica uma correspondência entre cada trecho e sua respectiva estação e coordenadas geográficas.

A coluna `periodicidade` possui apenas 2 valores distintos, compatíveis com as categorias previstas na especificação da base. Da mesma forma, `fonte` apresenta 2 valores distintos, correspondentes aos valores binários definidos para indicar se o trecho está associado a uma fonte hídrica.

A coluna `excluido` apresenta apenas 1 valor distinto. Esse resultado é relevante e será analisado posteriormente, pois indica que todos os registros da base possuem o mesmo valor nessa variável.

Por fim, `municipio_id` apresenta 8 valores distintos, assim como `municipio_nome`, indicando a presença de 8 municípios na base e uma correspondência entre os identificadores e seus respectivos nomes.

In [ ]:
df["excluido"].value_counts()

,count
excluido,
0,1694


Todos os trechos estão ativos

## 1.5 Colunas de texto

In [ ]:
colunas_texto = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Colunas de texto:")
print(colunas_texto)

Colunas de texto:
['trecho_nome', 'trecho_descricao', 'estacao', 'municipio_nome']


In [ ]:
for col in colunas_texto:

    print("=" * 70)
    print(f"COLUNA: {col}")
    print(f"Quantidade de valores únicos: {df[col].nunique(dropna=False)}")

    print("\nValores:")
    valores = df[col].value_counts(dropna=False)

    display(valores)

COLUNA: trecho_nome
Quantidade de valores únicos: 74

Valores:


,count
trecho_nome,
Manaíra (Quadra),34
Bessa,34
Bessa I,34
Manaíra (Residencia 1461),34
Manaíra (Verde mar pousada),33
Manaíra (Barramas),33
Bessa II,33
Tambaú Busto,33
Cabo Branco IV,33


COLUNA: trecho_descricao
Quantidade de valores únicos: 71

Valores:


,count
trecho_descricao,
Em frente a galeria de águas pluviais,64
Em frente a desembocadura da Lagoa,48
Em frente a desembocadura do Maceió do Bessa,34
Em frente a quadra de Manaíra,34
Em frente ao N° 1461 da Av. João Maurício,34
Rua Severino Nicolau de Melo,34
Final da Av. Gov. Flávio Ribeiro Coutinho,33
Em frente ao busto de Tamandaré,33
No final da AVENIDA MONSENHOR ODILON COUTINHO,33


COLUNA: estacao
Quantidade de valores únicos: 74

Valores:


,count
estacao,
06.02 B,34
06.01.1,34
06.01,34
06.02 B3,34
06.02 C1,33
06.02 C,33
06.01 A,33
06.03 A,33
06.04 C,33


COLUNA: municipio_nome
Quantidade de valores únicos: 8

Valores:


,count
municipio_nome,
João Pessoa,778
Cabedelo,278
Pitimbú,242
Lucena,162
Conde,90
Baia da Traição,69
Mataraca,38
Rio Tinto,37


## 1.6 Colunas categóricas

In [ ]:
colunas_categoricas = [
    "periodicidade",
    "fonte",
    "excluido"
]

for col in colunas_categoricas:

    print("=" * 70)

    print(
        df[col]
        .value_counts(dropna=False)
        .sort_index()
    )

periodicidade
1    1541
2     153
Name: count, dtype: int64
fonte
0    1462
1     232
Name: count, dtype: int64
excluido
0    1694
Name: count, dtype: int64


## 1.7 Colunas de datas

In [ ]:
display(
    df_ordenado[
        ["trecho_id", "analise_data", "periodicidade", "intervalo_dias"]
    ]
    .dropna(subset=["intervalo_dias"])
    .sort_values("intervalo_dias")
)

,trecho_id,analise_data,periodicidade,intervalo_dias
1,1,2026-05-14 13:43:29,2,0.000000
80,8,2026-05-14 13:43:58,2,0.000000
81,8,2026-05-14 13:43:58,2,0.000000
15,3,2026-05-14 13:43:33,2,0.000000
433,24,2026-05-14 12:13:04,1,0.000000
...,...,...,...,...
117,10,2026-07-29 14:47:54,2,40.181690
1470,67,2026-07-10 12:25:25,1,42.019167
47,5,2026-07-31 10:25:32,2,48.940775
1451,66,2026-07-10 12:24:10,1,56.875475


Foram identificados intervalos superiores ao esperado para a periodicidade cadastrada em alguns trechos. Essas ocorrências foram classificadas como possíveis inconsistências de frequência, não como datas inválidas, pois não há evidência suficiente para determinar que os registros estejam incorretos. Os dados foram preservados.

## 1.8 Tratamento das duplicidades

In [ ]:
duplicidade_trecho_data = (
    df.groupby(["trecho_id", "analise_data"])
      .size()
      .reset_index(name="qtd")
      .sort_values("qtd", ascending=False)
)

duplicidade_trecho_data[
    duplicidade_trecho_data["qtd"] > 1
].head(30)

,trecho_id,analise_data,qtd
100,11,2026-05-14 13:44:09,3
1430,68,2026-05-14 13:29:38,3
1506,72,2026-05-14 13:29:10,3
66,8,2026-05-14 13:43:58,3
1340,59,2026-05-14 13:42:48,2
1,1,2026-05-14 13:43:30,2
35,5,2026-05-14 13:43:48,2
1391,65,2026-05-14 13:42:28,2
1445,69,2026-05-14 13:42:04,2
312,21,2026-05-14 12:12:10,2


Existem vários casos em que o mesmo trecho_id possui 2 ou 3 registros com exatamente a mesma analise_data. Analisando entradas iguais exceto o analise_id:

In [ ]:
duplicatas_completas = df[
    df.duplicated(
        subset=[c for c in df.columns if c != "analise_id"],
        keep=False
    )
].sort_values(
    [c for c in df.columns if c != "analise_id"]
)

display(duplicatas_completas)

,analise_id,quantitativo,analise_data,trecho_id,trecho_nome,trecho_descricao,estacao,latitude,longitude,periodicidade,fonte,excluido,municipio_id,municipio_nome
199,13,0,2026-05-14 12:09:03,15,Ponta de Lucena,Na desembocadura da Camboa em Ponta de Lucena,04.01 A,-6.899897,-34.861600,1,0,0,4,Lucena
200,14,0,2026-05-14 12:09:03,15,Ponta de Lucena,Na desembocadura da Camboa em Ponta de Lucena,04.01 A,-6.899897,-34.861600,1,0,0,4,Lucena
227,19,0,2026-05-14 12:09:24,16,Gameleira,Em frente a desembocadura do Riacho Araçá,04.01 B,-6.920944,-34.863708,1,0,0,4,Lucena
228,20,0,2026-05-14 12:09:24,16,Gameleira,Em frente a desembocadura do Riacho Araçá,04.01 B,-6.920944,-34.863708,1,0,0,4,Lucena
281,29,0,2026-05-14 12:09:48,18,Costinha,No final da Rua Ubiratan Galvão,04.02,-6.964906,-34.855286,1,0,0,4,Lucena
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,383,0,2026-05-14 13:44:12,12,Oiteiro,Trechos próximos a foz do Rio Miriri,03.03,-6.833336,-34.909958,2,0,0,3,Rio Tinto
133,384,0,2026-05-14 13:44:13,12,Oiteiro,Trechos próximos a foz do Rio Miriri,03.03,-6.833336,-34.909958,2,0,0,3,Rio Tinto
134,385,0,2026-05-14 13:44:13,12,Oiteiro,Trechos próximos a foz do Rio Miriri,03.03,-6.833336,-34.909958,2,0,0,3,Rio Tinto
55,746,0,2026-05-14 14:55:57,6,Praia do Forte,No final da Rua Don Pedro II,02.01,-6.673269,-34.953075,2,0,0,2,Baia da Traição


In [ ]:
grupos_duplicados = (
    df[
        df.duplicated(
            subset=colunas_duplicidade,
            keep=False
        )
    ]
    .groupby(colunas_duplicidade, dropna=False)
    .agg(
        qtd_registros=("analise_id", "size"),
        analise_ids=("analise_id", list)
    )
    .reset_index()
)

display(
    grupos_duplicados[
        ["qtd_registros", "analise_ids"]
    ]
)

,qtd_registros,analise_ids
0,2,"[13, 14]"
1,2,"[19, 20]"
2,2,"[29, 30]"
3,2,"[39, 40]"
4,2,"[46, 47]"
5,2,"[48, 49]"
6,2,"[56, 57]"
7,2,"[58, 59]"
8,2,"[65, 66]"
9,3,"[183, 184, 185]"


As entradas duplicadas possuem ids sequenciais.

In [ ]:
grupos_duplicados = (
    df[df.duplicated(subset=colunas_duplicidade, keep=False)]
    .groupby(colunas_duplicidade, dropna=False)
    .size()
    .reset_index(name="qtd")
)

display(grupos_duplicados["qtd"].value_counts().sort_index())

,count
qtd,
2,51
3,4


Existem 51 grupos com 2 registros idênticos e 4 grupos com 3 registros dênticos:
- total: 55 grupos;
- total de linhas envolvidas: 51 × 2 + 4 × 3 = 114;
- 59 linhas a remover mantendo a primeria ocorrência de cada grupo.

In [ ]:
duplicados = df[
    df.duplicated(
        subset=colunas_duplicidade,
        keep=False
    )
]

resumo = (
    duplicados
    .groupby(colunas_duplicidade, dropna=False)
    .agg(
        qtd_registros=("analise_id", "size"),
        qtd_analise_id=("analise_id", "nunique")
    )
    .reset_index()
)

display(resumo)

,quantitativo,analise_data,trecho_id,trecho_nome,trecho_descricao,estacao,latitude,longitude,periodicidade,fonte,excluido,municipio_id,municipio_nome,qtd_registros,qtd_analise_id
0,0,2026-05-14 12:09:03,15,Ponta de Lucena,Na desembocadura da Camboa em Ponta de Lucena,04.01 A,-6.899897,-34.861600,1,0,0,4,Lucena,2,2
1,0,2026-05-14 12:09:24,16,Gameleira,Em frente a desembocadura do Riacho Araçá,04.01 B,-6.920944,-34.863708,1,0,0,4,Lucena,2,2
2,0,2026-05-14 12:09:48,18,Costinha,No final da Rua Ubiratan Galvão,04.02,-6.964906,-34.855286,1,0,0,4,Lucena,2,2
3,0,2026-05-14 12:12:10,21,Ponta de Mato,No final da Rua Nossa senhora dos Navegantes,05.02,-6.971361,-34.828506,1,0,0,5,Cabedelo,2,2
4,0,2026-05-14 12:12:26,22,Formosa,No final da Rua Monsenhor José Coutinho da silva,05.02 A,-6.978947,-34.827603,1,0,0,5,Cabedelo,2,2
5,0,2026-05-14 12:12:36,23,Areia Dourada,No final da Rua Projetada,05.02 B,-6.995175,-34.826500,1,0,0,5,Cabedelo,2,2
6,0,2026-05-14 12:13:04,24,Camboinha,No final da Rua Benício de Oliveira,05.03,-7.012900,-34.827728,1,0,0,5,Cabedelo,2,2
7,0,2026-05-14 12:13:18,25,Poço,No final da Rua Santa Cavalcante,05.04,-7.025753,-34.829764,1,0,0,5,Cabedelo,2,2
8,0,2026-05-14 12:13:46,26,Ponta de Campina,Em frente a galeria de águas pluviais,05.04 A,-7.032272,-34.831042,1,0,0,5,Cabedelo,2,2
9,0,2026-05-14 13:29:10,72,Riacho Eng. Velho,AVENIDA BEIRA MAR,F6,-7.476397,-34.809075,1,1,0,8,Pitimbú,3,3


Foram identificados 55 grupos de registros duplicados, totalizando 114 linhas. Os grupos apresentaram analise_id distintos e, em todos os casos, os demais atributos eram idênticos. Os IDs duplicados apresentaram ainda padrão sequencial na base. Para cada grupo foi mantida uma única ocorrência, resultando na remoção de 59 registros redundantes.

In [ ]:
colunas_duplicidade = [
    c for c in df.columns
    if c != "analise_id"
]

antes = len(df)

df_clean = df.drop_duplicates(
    subset=colunas_duplicidade,
    keep="first"
).copy()

depois = len(df_clean)

print("Antes:", antes)
print("Depois:", depois)
print("Removidos:", antes - depois)
print("analise_id duplicados:", df_clean["analise_id"].duplicated().sum())

Antes: 1694
Depois: 1635
Removidos: 59
analise_id duplicados: 0


# 2. Normalização

## 2.1 Municípios

In [ ]:
municipios = (
    df_clean[
        ["municipio_id", "municipio_nome"]
    ]
    .drop_duplicates()
    .sort_values("municipio_id")
    .reset_index(drop=True)
)

In [ ]:
print("Municípios:", len(municipios))

Municípios: 8


In [ ]:
print(
    "PK municipios duplicados:",
    municipios["municipio_id"].duplicated().sum()
)

PK municipios duplicados: 0


## 2.2 Trechos

In [ ]:
trechos = (
    df_clean[
        [
            "trecho_id",
            "municipio_id",
            "trecho_nome",
            "trecho_descricao",
            "estacao",
            "latitude",
            "longitude",
            "periodicidade",
            "fonte",
            "excluido"
        ]
    ]
    .drop_duplicates()
    .sort_values("trecho_id")
    .reset_index(drop=True)
)

In [ ]:
print("Trechos:", len(trechos))

Trechos: 74


In [ ]:
print(
    "PK trechos duplicaos:",
    trechos["trecho_id"].duplicated().sum()
)

PK trechos duplicaos: 0


In [ ]:
fk_municipio = ~trechos["municipio_id"].isin(
    municipios["municipio_id"]
)

print("FK municipio inválidas:", fk_municipio.sum())

FK municipio inválidas: 0


## 2.3 Análises

In [ ]:
analises = (
    df_clean[
        [
            "analise_id",
            "trecho_id",
            "analise_data",
            "quantitativo"
        ]
    ]
    .drop_duplicates()
    .sort_values("analise_id")
    .reset_index(drop=True)
)

In [ ]:
print("Análises:", len(analises))

Análises: 1635


In [ ]:
print(
    "PK analises duplicadas:",
    analises["analise_id"].duplicated().sum()
)

PK analises duplicadas: 0


In [ ]:
fk_trecho = ~analises["trecho_id"].isin(
    trechos["trecho_id"]
)

print("FK trecho inválidas:", fk_trecho.sum())

FK trecho inválidas: 0


## 2.4 Validação de PKs e FKs

In [ ]:
print("Base original/tratada:", len(df), "linhas")
print("Municípios:", len(municipios))
print("Trechos:", len(trechos))
print("Análises:", len(analises))

Base original/tratada: 1694 linhas
Municípios: 8
Trechos: 74
Análises: 1635


In [ ]:
print(municipios.shape)
print(trechos.shape)
print(analises.shape)

(8, 2)
(74, 10)
(1635, 4)


## 2.5 Exportação dos CSVs

In [ ]:
municipios.to_csv(
    "municipios.csv",
    index=False,
    encoding="utf-8-sig"
)

trechos.to_csv(
    "trechos.csv",
    index=False,
    encoding="utf-8-sig"
)

analises.to_csv(
    "analises.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Conclusões do tratamento e normalização

A análise da qualidade dos dados permitiu identificar e avaliar possíveis inconsistências presentes na base original. Não foram identificados valores ausentes nas colunas analisadas, e os valores das variáveis categóricas apresentaram-se compatíveis com a especificação fornecida.

Na análise das datas, foram observados alguns intervalos superiores aos esperados para a periodicidade cadastrada. Essas ocorrências foram tratadas como possíveis inconsistências de frequência, e não como datas inválidas, pois não havia evidências suficientes para determinar que os registros estivessem incorretos. Dessa forma, os dados foram preservados.

Também foram identificados 55 grupos de registros com informações idênticas em todas as colunas, exceto `analise_id`, totalizando 114 linhas envolvidas. Em cada grupo, os registros apresentavam `analise_id` distintos e os demais atributos eram iguais. Para eliminar a redundância, foi mantida uma única ocorrência de cada grupo, resultando na remoção de 59 registros. A base tratada passou, portanto, de 1.694 para 1.635 registros.

Após o tratamento, a base foi normalizada em três entidades: `municipios`, `trechos` e `analises`. A tabela `municipios` contém 8 registros, `trechos` contém 74 registros e `analises` contém 1.635 registros.

O modelo estabelece uma relação de um município para muitos trechos e de um trecho para muitas análises. Foram utilizadas `municipio_id`, `trecho_id` e `analise_id` como chaves primárias das respectivas tabelas, enquanto `municipio_id` em `trechos` e `trecho_id` em `analises` atuam como chaves estrangeiras.

As validações realizadas não identificaram duplicidades nas chaves primárias nem referências inválidas nas chaves estrangeiras. Dessa forma, as tabelas resultantes estão preparadas para serem utilizadas na etapa seguinte de persistência em SQLite e implementação das regras de classificação.